# Main section

In [1]:
# Imports
import numpy as np
import torch
import os


### Load and save the models

In [ ]:

from transformers import AutoTokenizer, BertForSequenceClassification
import logging
logger = logging.getLogger(__name__)
force_load = True

# tries to load model saved locally, otherwise fetch from huggingface
try:
    if(not force_load):
        model = BertForSequenceClassification.from_pretrained("models/bert")
        tokenizer = AutoTokenizer.from_pretrained("models/tokenizer")
        print("loaded locally")
    else:
        raise Exception
    # inputs = tokenizer("Hello, my dog is cute", return_tensors="pt")
    # outputs = model(**inputs)

    # prediction_logits = outputs.prediction_logits
    # seq_relationship_logits = outputs.seq_relationship_logits

except Exception as e:
    logger.exception(e)
    print("loading tokenizer from hugging face")
    tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-cased")
    tokenizer.save_pretrained("models/tokenizer")

    print("loading model from hugging face")
    model = BertForSequenceClassification.from_pretrained("google-bert/bert-base-cased")
    model.save_pretrained("models/bert")


Traceback (most recent call last):
  File "/var/folders/bx/nfgqm9dn79lfw_rvrhvq3dl00000gn/T/ipykernel_64058/3285787988.py", line 13, in <module>
    raise Exception
Exception


loading tokenizer from hugging face
loading model from hugging face


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9131.72it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from th

### Data preprocessing

In [58]:
import pandas as pd
from pandas import DataFrame

df = pd.read_csv('dataset/fake_job_postings.csv')
headings = df.columns.values
print(headings)
headings.shape

<StringArray>
[             'job_id',               'title',            'location',
          'department',        'salary_range',     'company_profile',
         'description',        'requirements',            'benefits',
       'telecommuting',    'has_company_logo',       'has_questions',
     'employment_type', 'required_experience',  'required_education',
            'industry',            'function',          'fraudulent']
Length: 18, dtype: str


(18,)

In [60]:
print(df.isna().sum())
print(df.shape)

job_id                     0
title                      0
location                 346
department             11547
salary_range           15012
company_profile         3308
description                1
requirements            2696
benefits                7212
telecommuting              0
has_company_logo           0
has_questions              0
employment_type         3471
required_experience     7050
required_education      8105
industry                4903
function                6455
fraudulent                 0
dtype: int64
(17880, 18)


As we can see here there are columns that are empty for most entries, therefore we decided to drop columns where there are more than 1000 empty entires to make the data easier to work with.

In [61]:
# check columns with too many missing points
count = df.isna().sum().to_list()
to_drop = [col for col, n in zip(df.columns, count) if n > 1000]
df_filtered = df.drop(columns=to_drop)
df_filtered.columns

Index(['job_id', 'title', 'location', 'description', 'telecommuting',
       'has_company_logo', 'has_questions', 'fraudulent'],
      dtype='str')

In [ ]:
from typing import Any

from torch.utils.data import Dataset

class Real_and_Fake_Jobs(Dataset):
    def __init__(self) -> None:
        super().__init__()

    def __getitem__(self, index: int) -> Any:
        return super().__getitem__(index)

    def __len__(self):
        pass



### Problem: Input size exceeeds max sequence length
To fix this, 

In [141]:

# text = tokenizer.tokenize(df_filtered.iloc[5]["description"])
# ignore job_id, fradument
text_cols = ['title','location', 'telecommuting', 'has_company_logo', 'has_questions', 'description']
# convert binary entries to text
flag_text = {
    "telecommuting": {1: "telecommuting allowed", 0: "telecommuting not allowed"},
    "has_company_logo": {1: "has company logo", 0: "no company logo"},
    "has_questions": {1: "has questions", 0: "no questions"},
}
max_seq_length = 512
for col, mapping in flag_text.items():
    df_filtered[col] = df_filtered[col].map(mapping)

parts = []
for row in df_filtered[text_cols].values:
    field = []
    for j in range(len(row)):
        if pd.isna(row[j]) or str(row[j]).strip() == "":
            row[j] = "None"
        # print(f'{text_cols[j]}: {row[j]}')
        field.append(f'{text_cols[j]}: {row[j]}')
    parts.append(" ".join(field))

tokens = tokenizer.tokenize(parts)
parts

# df_text = df_filtered[text_cols].fillna("").astype(str).agg(" [SEP] ".join, axis=1)

# print(df_filtered.iloc[0],'\n',df_text[0])



['title: Marketing Intern location: US, NY, New York telecommuting: None has_company_logo: None has_questions: None description: Food52, a fast-growing, James Beard Award-winning online food community and crowd-sourced and curated recipe hub, is currently interviewing full- and part-time unpaid interns to work in a small team of editors, executives, and developers in its New York City headquarters.Reproducing and/or repackaging existing Food52 content for a number of partner sites, such as Huffington Post, Yahoo, Buzzfeed, and more in their various content management systemsResearching blogs and websites for the Provisions by Food52 Affiliate ProgramAssisting in day-to-day affiliate program support, such as screening affiliates and assisting in any affiliate inquiriesSupporting with PR &amp; Events when neededHelping with office administrative work, such as filing, mailing, and preparing for meetingsWorking with developers to document bugs and suggest improvements to the siteSupporting

In [72]:
for name, p in model.named_parameters():
    print(name)

bert.embeddings.word_embeddings.weight
bert.embeddings.position_embeddings.weight
bert.embeddings.token_type_embeddings.weight
bert.embeddings.LayerNorm.weight
bert.embeddings.LayerNorm.bias
bert.encoder.layer.0.attention.self.query.weight
bert.encoder.layer.0.attention.self.query.bias
bert.encoder.layer.0.attention.self.key.weight
bert.encoder.layer.0.attention.self.key.bias
bert.encoder.layer.0.attention.self.value.weight
bert.encoder.layer.0.attention.self.value.bias
bert.encoder.layer.0.attention.output.dense.weight
bert.encoder.layer.0.attention.output.dense.bias
bert.encoder.layer.0.attention.output.LayerNorm.weight
bert.encoder.layer.0.attention.output.LayerNorm.bias
bert.encoder.layer.0.intermediate.dense.weight
bert.encoder.layer.0.intermediate.dense.bias
bert.encoder.layer.0.output.dense.weight
bert.encoder.layer.0.output.dense.bias
bert.encoder.layer.0.output.LayerNorm.weight
bert.encoder.layer.0.output.LayerNorm.bias
bert.encoder.layer.1.attention.self.query.weight
bert.enc